# 04 — Which subspace carries the digitStage 3. Decoding said the information is there. It did not say what carries it.The naive version of the question is "which neurons?", and that is the wrong unit. A population code is distributed and redundant: stimulus identity is carried by a **shared subspace**, a coordinated pattern across many channels, not by a list of important cells.> **Required framing, to be stated explicitly in your report:** selection and ablation here establish causal relevance **for the decoder, not for the circuit**. The data are fixed recordings and nothing is intervened upon in the biological system.

In [ ]:
%load_ext autoreload%autoreload 2import numpy as npimport matplotlib.pyplot as pltimport pandas as pdfrom compbio2026 import data, decoding, plotting, selectionplotting.apply_style()rng = np.random.default_rng(2026)shd = data.load("train")idx = np.flatnonzero(shd.english_mask())idx = rng.choice(idx, size=min(2000, len(idx)), replace=False)X, y, t = data.build_design_matrix(shd, bin_ms=20.0, t_max_ms=800.0, smooth_ms=10.0, trials=idx)print(X.shape)

## The heuristics firstAlways run the simple things before the sophisticated ones. If a firing-rate threshold matches the structured method, **that is a result**, not a failure.

In [ ]:
k = 50sel = {    "random": selection.select_random(X.shape[1], k, rng),    "firing_rate": selection.select_by_rate(X, k),    "variance": selection.select_by_variance(X, k),    "discriminability": selection.select_by_discriminability(X, y, k),}fig, ax = plt.subplots(figsize=(9, 2.6))for i, (name, ch) in enumerate(sel.items()):    ax.scatter(ch, np.full(len(ch), i), s=18, color=plotting.PALETTE[i], linewidths=0)ax.set_yticks(range(len(sel)))ax.set_yticklabels(list(sel))ax.set_xlabel("Channel (tonotopic)")ax.set_xlim(0, X.shape[1])ax.set_title(f"Which {k} channels each method picks")

Do the methods agree? Where the selected channels sit along the tonotopic axis is itself informative — clustering in one frequency band means something different from a spread.

## Sparse group lassoEach **group** is one cochlear channel across all of its time bins. The group penalty selects channels; within-group sparsity keeps the temporal support tight. The output is a short list of channels — a candidate basis for the subspace, not a ranking of neurons.

In [ ]:
try:    ch_sgl, norms = selection.sparse_group_lasso_channels(X, y, group_reg=0.05, l1_reg=0.01, k=k)    sel["sparse_group_lasso"] = ch_sgl    fig, ax = plt.subplots(figsize=(9, 2.8))    ax.plot(norms, color=plotting.ACCENT, lw=1)    ax.scatter(ch_sgl, norms[ch_sgl], color=plotting.PALETTE[1], s=14, zorder=3, linewidths=0)    ax.set_xlabel("Channel (tonotopic)")    ax.set_ylabel("Group norm")    ax.set_title(f"Sparse group lasso: {len(ch_sgl)} channels selected")except ImportError:    print("group-lasso not installed:  pip install group-lasso")

**Tune, do not guess.** `group_reg` controls how many channels survive. Sweep it and report the *path* — how accuracy and selection size trade off — rather than one arbitrary point on it.

In [ ]:
path = []for gr in [0.01, 0.03, 0.05, 0.1, 0.2]:    try:        ch, _ = selection.sparse_group_lasso_channels(X, y, group_reg=gr, l1_reg=0.01)        r = selection.ablate(X, y, ch)        path.append({"group_reg": gr, "n_selected": len(ch), "kept_only": r["kept"], "full": r["full"]})    except ImportError:        breakpd.DataFrame(path).round(3) if path else "group-lasso not available"

## Ablation — the necessity testA set the decoder *can* use is not a set it *needs*. Remove the selection, refit from scratch, and measure the drop.

In [ ]:
results = {name: {**selection.ablate(X, y, ch), "channels": ch} for name, ch in sel.items()}fig, ax = plt.subplots(figsize=(8, 4))plotting.ablation_bars(results, ax=ax)ax.set_title(f"Full / selection only / selection removed  (k = {k})")pd.DataFrame({n: {k2: v for k2, v in r.items() if k2 != "channels"} for n, r in results.items()}).T.round(3)

**Read the `compensation` column.** It is the gap between full accuracy and accuracy after removing the selection. In a redundant code the remaining population compensates and that gap is small — which means the "important" channels were never necessary.**That number is the most informative result in this project.** A small compensation is not a failed experiment; it is a quantitative statement about redundancy, and it is the honest answer to "which neurons matter?".

## How much of the population do you need?Sweep `k` and watch how quickly accuracy from the selection alone saturates.

In [ ]:
rows = []for kk in [5, 10, 25, 50, 100, 200, 400]:    if kk > X.shape[1]:        break    ch = selection.select_by_discriminability(X, y, kk)    r = selection.ablate(X, y, ch)    rows.append({"k": kk, "kept_only": r["kept"], "ablated": r["ablated"],                 "full": r["full"], "compensation": r["compensation"]})curve = pd.DataFrame(rows)fig, ax = plt.subplots(figsize=(6, 3.6))ax.plot(curve["k"], curve["kept_only"], marker="o", ms=4, label="selection only", color=plotting.ACCENT)ax.plot(curve["k"], curve["ablated"], marker="s", ms=4, label="selection removed", color=plotting.PALETTE[1])ax.axhline(curve["full"].iloc[0], ls=":", lw=1, color=plotting.INK_MUTED)ax.axhline(decoding.chance_level(y), ls="--", lw=1, color=plotting.INK_MUTED)ax.set_xscale("log")ax.set_xlabel("Channels selected (k)")ax.set_ylabel("Accuracy")ax.legend()curve.round(3)

---## Exercises1. **Is the subspace shared across digits, or per digit?** Run one-vs-rest selection for each digit and compare the selected sets. Overlapping or orthogonal is an empirical question with a real answer.2. **Random-subset control.** Compare selection-only accuracy against `k` *random* channels, repeated. How much of the selection's performance is just "having k channels"?3. **Tonotopic structure.** Are the selected channels clustered in frequency? Compare against a shuffled-channel null.4. **Temporal support.** Sparse group lasso also selects *when* within each channel. Plot the selected time bins. Does the decoder rely on onsets, offsets, or the sustained portion?5. **Write the caveat.** In your own words, one paragraph: why does ablation here not license a claim about the biological circuit? What experiment would?